# Task 6: Full Predoc Simulation

**Time target:** 5-6 hours (TIMED)  
**Date attempted:**  
**Start time:**  
**End time:**  
**Actual time:**  

See `instructions.md` for full task description, time allocation, and questions.

**Record your time at each phase transition.**

## 1. Setup & Data Generation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

In [ ]:
# --- DATA GENERATION (do not modify) ---
np.random.seed(2025)

# State setup
states_full = [
    'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado',
    'Connecticut', 'Delaware', 'Florida', 'Georgia', 'Hawaii', 'Idaho',
    'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana',
    'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota',
    'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada',
    'New Hampshire', 'New Jersey', 'New Mexico', 'New York',
    'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon',
    'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota',
    'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington',
    'West Virginia', 'Wisconsin', 'Wyoming'
]
states_abbrev = [
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA',
    'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
    'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
    'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
    'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY'
]
state_map = dict(zip(states_full, states_abbrev))
treated_states = states_full[:12]  # First 12 states expanded EITC

years = list(range(2015, 2023))
n_per_state_year = 125

# --- Dataset 1: individual_survey.csv ---
survey_rows = []
pid = 0
for state in states_full:
    is_treated = state in treated_states
    for year in years:
        for _ in range(n_per_state_year):
            pid += 1
            age = int(np.random.normal(38, 10))
            sex = np.random.choice(['Female', 'Male'], p=[0.52, 0.48])
            marital = np.random.choice(['Married', 'Single', 'Divorced', 'Widowed'], p=[0.4, 0.35, 0.2, 0.05])
            n_children = np.random.choice([0, 1, 2, 3, 4], p=[0.3, 0.25, 0.25, 0.15, 0.05])
            edu = np.random.choice(['Less than HS', 'High School', 'Some College', 'Bachelors', 'Graduate'],
                                   p=[0.1, 0.25, 0.3, 0.25, 0.1])
            
            # Employment depends on treatment for single mothers
            is_single_mother = (sex == 'Female' and marital != 'Married' and n_children > 0)
            post = year >= 2018
            base_emp_prob = 0.65 if is_single_mother else 0.75
            if is_single_mother and is_treated and post:
                base_emp_prob += 0.06  # True treatment effect: +6pp
            employed = int(np.random.random() < base_emp_prob)
            hours = int(np.random.normal(38, 8)) if employed else 0
            earnings = int(np.random.lognormal(10.5, 0.6)) if employed else 0
            
            # Messy state names: ~30% use abbreviation
            state_val = state_map[state] if np.random.random() < 0.3 else state
            
            # Messy earnings: ~5% have $ and commas
            earnings_val = f'${earnings:,}' if np.random.random() < 0.05 else str(earnings)
            
            row = {
                'person_id': pid,
                'state': state_val,
                'year': year,
                'age': age,
                'sex': sex,
                'marital_status': marital,
                'num_children': n_children,
                'education': edu,
                'employed': employed,
                'hours_worked': hours,
                'annual_earnings': earnings_val,
            }
            survey_rows.append(row)

individual_survey = pd.DataFrame(survey_rows)

# Add ~0.5% negative ages (data entry errors)
bad_idx = individual_survey.sample(frac=0.005).index
individual_survey.loc[bad_idx, 'age'] = -individual_survey.loc[bad_idx, 'age'].abs()

# Add ~0.3% employed=1 but hours=0 (inconsistency)
incon_idx = individual_survey[individual_survey['employed'] == 1].sample(frac=0.003).index
individual_survey.loc[incon_idx, 'hours_worked'] = 0

# Add ~50 duplicate rows
dupes = individual_survey.sample(50)
individual_survey = pd.concat([individual_survey, dupes], ignore_index=True)

# --- Dataset 2: state_policy.csv ---
policy_rows = []
for state in states_full:
    is_treated = state in treated_states
    policy_rows.append({
        'state_name': state,
        'eitc_expansion': int(is_treated),
        'expansion_year': 2018 if is_treated else np.nan,
        'state_population': int(np.random.uniform(500000, 20000000)),
        'median_hh_income': int(np.random.normal(60000, 12000)),
    })
state_policy = pd.DataFrame(policy_rows)
# Add conflicting expansion_year for 2 states
state_policy.loc[state_policy['state_name'] == 'Alaska', 'expansion_year'] = 2017
state_policy.loc[state_policy['state_name'] == 'Arizona', 'expansion_year'] = 2019

# --- Dataset 3: state_economics.csv ---
econ_rows = []
for state in states_full:
    base_unemp = np.random.uniform(0.03, 0.08)
    base_gdp = np.random.uniform(40000, 80000)
    base_mw = np.random.choice([7.25, 9.0, 10.0, 12.0, 15.0])
    for year in years:
        # Some years as strings with whitespace
        year_val = f' {year} ' if np.random.random() < 0.08 else year
        econ_rows.append({
            'state': state,
            'year': year_val,
            'unemployment_rate': round(base_unemp + np.random.normal(0, 0.005), 4),
            'gdp_per_capita': round(base_gdp + 1000 * (year - 2015) + np.random.normal(0, 2000), 0),
            'min_wage': base_mw + 0.5 * max(0, year - 2017),
        })
state_economics = pd.DataFrame(econ_rows)

print("individual_survey:", individual_survey.shape)
print("state_policy:", state_policy.shape)
print("state_economics:", state_economics.shape)
print("\n--- Intentional data issues ---")
print(f"Negative ages: {(individual_survey['age'] < 0).sum()}")
print(f"Earnings with $ signs: {individual_survey['annual_earnings'].str.contains(r'\$', na=False).sum()}")
print(f"Duplicate rows: ~50 added")
print(f"Mixed state formats (full/abbrev): yes")
print(f"Years with whitespace in state_economics: {(state_economics['year'].apply(lambda x: isinstance(x, str))).sum()}")
print(f"Conflicting expansion years: Alaska=2017, Arizona=2019")

---
## Phase 1: Data Assessment & Cleaning (target: 90 min)

**Time started:**

In [ ]:
# Assess individual_survey


In [ ]:
# Assess state_policy


In [ ]:
# Assess state_economics


In [ ]:
# Clean individual_survey


In [ ]:
# Harmonize state identifiers


In [ ]:
# Clean state_economics (year whitespace, etc.)


In [ ]:
# Handle conflicting expansion years in state_policy


In [ ]:
# Merge all datasets


---
## Phase 2: Sample Construction & Descriptives (target: 45 min)

**Time started:**

In [ ]:
# Construct analysis sample: single mothers, aged 20-55


In [ ]:
# Balance table: treated vs control, pre-period


In [ ]:
# Parallel trends plot


---
## Phase 3: Analysis (target: 60 min)

**Time started:**

In [ ]:
# DiD Model 1: Basic with controls and FEs


In [ ]:
# DiD Model 2: Add state-level economic controls


In [ ]:
# DiD Model 3: Additional specification


In [ ]:
# Regression table: 3 specifications side by side


---
## Phase 4: Visualization (target: 45 min)

**Time started:**

In [ ]:
# (a) Parallel trends plot (publication quality)


In [ ]:
# (b) Coefficient/event-study plot (if time permits)


In [ ]:
# (c) Additional figure


---
## Phase 5: Research Memo (target: 30 min)

**Time started:**

### Research Question and Motivation

[What is the effect of state EITC expansions on labor force participation among single mothers?]

### Data Sources and Sample Construction

[Describe the three datasets, cleaning steps, and sample restrictions.]

### Empirical Strategy

[Describe the DiD approach, identification assumptions, and what would violate them.]

### Key Results

[Report the treatment effect estimate, significance, and magnitude in practical terms.]

### Limitations and Threats to Validity

[Discuss: parallel trends, SUTVA, anticipation effects, composition changes, etc.]

### Further Research

[One concrete suggestion for extending this analysis.]

---
## Post-Task Review

**Total time:**  

**What went well:**  

**What to improve:**  

**Time allocation reflection:** Did you spend time where it mattered most?